# 파인튜닝된 rec 모델 실이미지 테스트 (export 없이 raw checkpoint로 det+rec 체이닝)

런타임을 GPU로: `런타임 > 런타임 유형 변경 > T4 GPU`.

학습 노트북(`colab_train.ipynb`)에서 이미 구글드라이브에 `output/` 백업해뒀다는 전제.

export_model.py / paddleocr / paddlex 전부 안 씀 — 학습(tools/train.py)때와 똑같은
`ppocr` 패키지 API로 det+rec을 직접 체이닝(`tools/infer_raw_pipeline.py`).
export 경로에서 자꾸 터지던 버전/PIR 문제 자체가 안 생김.

In [ ]:
# 1. 드라이브 마운트 + 의존성 설치 (paddleocr/paddlex 불필요 — ppocr 패키지만 씀)
# paddlepaddle-gpu(1.4GB, bcebos 느림)는 드라이브에 캐싱 — 최초 1회만 다운로드,
# 그 다음 세션부턴 드라이브에서 바로 설치(네트워크 안 타서 느림/타임아웃 문제 자체가 없어짐)
from google.colab import drive
drive.mount('/content/drive')

import glob, os
WHEEL_DIR = '/content/drive/MyDrive/paddle_wheels'
os.makedirs(WHEEL_DIR, exist_ok=True)
existing = glob.glob(f'{WHEEL_DIR}/paddlepaddle_gpu-3.3.1*.whl')
if existing:
    print(f'캐시 사용: {existing[0]}')
    !pip install "{existing[0]}" -q
else:
    print('캐시 없음 - 최초 1회 다운로드 (다음부턴 훨씬 빨라짐)')
    !pip download --timeout 300 --retries 5 paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/ -d {WHEEL_DIR} --no-deps
    existing = glob.glob(f'{WHEEL_DIR}/paddlepaddle_gpu-3.3.1*.whl')
    !pip install "{existing[0]}" -q

!pip install lmdb scikit-image albumentations opencv-python pyclipper shapely rapidfuzz pyyaml -q

In [ ]:
# 2. PaddleOCR 클론(--depth 1) + 백업된 rec checkpoint 복사 + det raw 가중치 다운로드
!git clone -q --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!cp -r /content/drive/MyDrive/PP-OCRv6_small_rec_finetune_output ./output
!mkdir -p pretrain_weights
!wget -q https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_mobile_det_pretrained.pdparams -O pretrain_weights/PP-OCRv5_mobile_det_pretrained.pdparams
!ls output/PP-OCRv6_small_rec_finetune
!ls pretrain_weights

In [ ]:
%%writefile tools/infer_raw_pipeline.py
"""
export 없이 raw checkpoint로 det->crop->rec 파이프라인 (회전판단은 rec 자체 신뢰도로)
"""

import argparse
import json
import os
import sys

__dir__ = os.path.dirname(os.path.abspath(__file__))
sys.path.append(__dir__)
sys.path.insert(0, os.path.abspath(os.path.join(__dir__, "..")))
os.environ["FLAGS_allocator_strategy"] = "auto_growth"

import cv2
import numpy as np
import paddle
import yaml
from PIL import Image, ImageDraw, ImageFont

from ppocr.data import create_operators, transform
from ppocr.modeling.architectures import build_model
from ppocr.postprocess import build_post_process
from ppocr.utils.save_load import load_model
from ppocr.utils.utility import get_image_file_list


def get_rotate_crop_image(img, points):
    points = points.astype(np.float32)
    img_crop_width = int(max(
        np.linalg.norm(points[0] - points[1]), np.linalg.norm(points[2] - points[3])))
    img_crop_height = int(max(
        np.linalg.norm(points[0] - points[3]), np.linalg.norm(points[1] - points[2])))
    pts_std = np.float32([
        [0, 0], [img_crop_width, 0], [img_crop_width, img_crop_height], [0, img_crop_height]])
    M = cv2.getPerspectiveTransform(points, pts_std)
    dst = cv2.warpPerspective(
        img, M, (img_crop_width, img_crop_height),
        borderMode=cv2.BORDER_REPLICATE, flags=cv2.INTER_CUBIC)
    h, w = dst.shape[:2]
    if h * 1.0 / max(w, 1) >= 1.5:
        dst = np.rot90(dst)
    return dst


def load_yaml(path):
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f)


def build_det(det_config_path, det_weights):
    config = load_yaml(det_config_path)
    config["Global"]["pretrained_model"] = det_weights
    global_config = config["Global"]
    model = build_model(config["Architecture"])
    load_model(config, model)
    model.eval()
    post_process_class = build_post_process(config["PostProcess"])
    transforms = []
    for op in config["Eval"]["dataset"]["transforms"]:
        op_name = list(op)[0]
        if "Label" in op_name:
            continue
        elif op_name == "KeepKeys":
            op[op_name]["keep_keys"] = ["image", "shape"]
        transforms.append(op)
    ops = create_operators(transforms, global_config)
    return model, post_process_class, ops


def build_rec(rec_config_path, rec_weights):
    config = load_yaml(rec_config_path)
    config["Global"]["pretrained_model"] = rec_weights
    global_config = config["Global"]
    post_process_class = build_post_process(config["PostProcess"], global_config)

    if hasattr(post_process_class, "character"):
        char_num = len(getattr(post_process_class, "character"))
        if config["Architecture"]["Head"]["name"] == "MultiHead":
            out_channels_list = {
                "CTCLabelDecode": char_num,
                "SARLabelDecode": char_num + 2,
                "NRTRLabelDecode": char_num + 3,
            }
            config["Architecture"]["Head"]["out_channels_list"] = out_channels_list
        else:
            config["Architecture"]["Head"]["out_channels"] = char_num

    model = build_model(config["Architecture"])
    load_model(config, model)
    model.eval()

    transforms = []
    for op in config["Eval"]["dataset"]["transforms"]:
        op_name = list(op)[0]
        if "Label" in op_name:
            continue
        elif op_name == "RecResizeImg":
            op[op_name]["infer_mode"] = True
        elif op_name == "KeepKeys":
            op[op_name]["keep_keys"] = ["image"]
        transforms.append(op)
    global_config["infer_mode"] = True
    ops = create_operators(transforms, global_config)
    return model, post_process_class, ops


@paddle.no_grad()
def run_det(model, post_process_class, ops, img_bytes):
    data = {"image": img_bytes}
    batch = transform(data, ops)
    images = np.expand_dims(batch[0], axis=0)
    shape_list = np.expand_dims(batch[1], axis=0)
    images = paddle.to_tensor(images)
    preds = model(images)
    post_result = post_process_class(preds, shape_list)
    return post_result[0]["points"]


@paddle.no_grad()
def run_rec(model, post_process_class, ops, crop_bgr):
    ok, buf = cv2.imencode(".png", crop_bgr)
    data = {"image": buf.tobytes()}
    batch = transform(data, ops)
    images = np.expand_dims(batch[0], axis=0)
    images = paddle.to_tensor(images)
    preds = model(images)
    post_result = post_process_class(preds)
    return post_result[0][0], float(post_result[0][1])


def rotate_crop(img, angle):
    """angle: 0/90/180/270(시계방향). 별도 방향분류 모델 없이 cv2.rotate만 씀."""
    if angle == 0:
        return img
    if angle == 90:
        return cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    if angle == 180:
        return cv2.rotate(img, cv2.ROTATE_180)
    if angle == 270:
        return cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
    raise ValueError(angle)


def resolve_orientation(rec_model, rec_post, rec_ops, crops, flip_vote_thresh=0.5):
    """방향분류 모델 없이 rec 자체 신뢰도로 회전 판단.
    1단계: 페이지 단위 180도 다수결(crop 전체가 같이 뒤집혀있다는 전제).
    2단계: crop별 0/90/270 중 rec 신뢰도 최고인 방향 채택(세로쓰기 개별 대응)."""
    if not crops:
        return [], [], False

    flip_votes = 0
    zero_results = []
    for crop in crops:
        t0, s0 = run_rec(rec_model, rec_post, rec_ops, crop)
        t180, s180 = run_rec(rec_model, rec_post, rec_ops, rotate_crop(crop, 180))
        zero_results.append((t0, s0))
        if s180 > s0:
            flip_votes += 1
    page_flip = (flip_votes / len(crops)) >= flip_vote_thresh

    final_crops, final_results = [], []
    for crop, (t0, s0) in zip(crops, zero_results):
        base = rotate_crop(crop, 180) if page_flip else crop
        if page_flip:
            best_text, best_score, best_crop = None, -1, None
            candidates = [0, 90, 270]
        else:
            best_text, best_score, best_crop = t0, s0, base
            candidates = [90, 270]
        for ang in candidates:
            c = rotate_crop(base, ang)
            t, s = run_rec(rec_model, rec_post, rec_ops, c)
            if s > best_score:
                best_text, best_score, best_crop = t, s, c
        final_crops.append(best_crop)
        final_results.append((best_text, best_score))

    return final_crops, final_results, page_flip


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--det_config", required=True)
    ap.add_argument("--det_weights", required=True)
    ap.add_argument("--rec_config", required=True)
    ap.add_argument("--rec_weights", required=True)
    ap.add_argument("--img_dir", required=True)
    ap.add_argument("--out_dir", default="vis_out_raw")
    ap.add_argument("--score_thresh", type=float, default=0.5)
    ap.add_argument("--flip_vote_thresh", type=float, default=0.5,
                     help="페이지 전체 180도 보정을 적용할 최소 득표비율")
    args = ap.parse_args()

    print("det 모델 로드 중...")
    det_model, det_post, det_ops = build_det(args.det_config, args.det_weights)
    print("rec 모델 로드 중...")
    rec_model, rec_post, rec_ops = build_rec(args.rec_config, args.rec_weights)

    os.makedirs(args.out_dir, exist_ok=True)
    images = get_image_file_list(args.img_dir)
    print(f"대상: {len(images)}장")

    try:
        font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
        assert os.path.exists(font_path)
    except AssertionError:
        font_path = None

    for path in images:
        with open(path, "rb") as f:
            img_bytes = f.read()
        boxes = run_det(det_model, det_post, det_ops, img_bytes)

        src_img = cv2.imread(path)
        pil_img = Image.open(path).convert("RGB")
        draw = ImageDraw.Draw(pil_img)
        size = max(14, pil_img.width // 60)
        font = ImageFont.truetype(font_path, size) if font_path else ImageFont.load_default()

        raw_crops = []
        pts_list = []
        for box in boxes:
            pts = np.array(box, dtype=np.float32)
            crop = get_rotate_crop_image(src_img, pts)
            if crop.size == 0:
                continue
            raw_crops.append(crop)
            pts_list.append(pts)

        _, results, page_flip = resolve_orientation(
            rec_model, rec_post, rec_ops, raw_crops, args.flip_vote_thresh)

        dets = []
        for pts, (text, score) in zip(pts_list, results):
            dets.append({"text": text, "score": round(score, 3), "poly": pts.tolist()})
            if score < args.score_thresh:
                continue

            pts_tuple = [tuple(p) for p in pts]
            draw.polygon(pts_tuple, outline=(255, 0, 0), width=2)
            x, y = min(p[0] for p in pts_tuple), min(p[1] for p in pts_tuple)
            label = f"{text} ({score:.2f})"
            tb = draw.textbbox((x, y), label, font=font)
            draw.rectangle([tb[0] - 2, tb[1] - 2, tb[2] + 2, tb[3] + 2], fill=(255, 255, 160))
            draw.text((x, y), label, fill=(200, 0, 0), font=font)

        stem = os.path.splitext(os.path.basename(path))[0]
        pil_img.save(os.path.join(args.out_dir, f"{stem}_vis.png"))
        with open(os.path.join(args.out_dir, f"{stem}.json"), "w", encoding="utf-8") as f:
            json.dump({"image": os.path.basename(path), "page_flip": page_flip,
                      "detections": dets}, f, indent=2, ensure_ascii=False)
        print(f"  {os.path.basename(path)}: {len(dets)}개 검출 (page_flip={page_flip})")

    print("완료")


if __name__ == "__main__":
    main()

In [ ]:
# 4. 테스트 이미지 업로드 (로컬 zip 선택 — data/generated/test_imgs_colab.zip)
# ※ 이 zip은 리팩토링 때 삭제됨. 필요하면 테스트용 실이미지 몇 장을 zip으로 다시 묶어 올릴 것.
from google.colab import files
uploaded = files.upload()
!mkdir -p test_imgs
!unzip -q test_imgs_colab.zip -d test_imgs
!ls test_imgs | head

In [ ]:
# 5. det(기존 v5, raw checkpoint) + rec(우리 파인튜닝, raw checkpoint) 파이프라인 실행
# rec_config는 원본 레포에 이미 있는 PP-OCRv6_small_rec.yml 그대로 씀 — 우리가 학습 때
# 따로 만든 finetune yml은 Global(에폭수/배치크기 등) 학습설정만 바꾼 거라 추론엔 안 씀.
# Architecture/PostProcess/Eval(전처리)은 원본 config와 동일해서 그거 그대로 참조하면 됨.
!python tools/infer_raw_pipeline.py \
  --det_config configs/det/PP-OCRv5/PP-OCRv5_mobile_det.yml \
  --det_weights pretrain_weights/PP-OCRv5_mobile_det_pretrained \
  --rec_config configs/rec/PP-OCRv6/PP-OCRv6_small_rec.yml \
  --rec_weights output/PP-OCRv6_small_rec_finetune/best_accuracy \
  --img_dir test_imgs \
  --out_dir vis_out_raw

In [ ]:
# 6. 결과 zip으로 다운로드
!zip -r -q vis_out_raw.zip vis_out_raw
from google.colab import files
files.download('vis_out_raw.zip')